In [1]:
import torch
import torch.nn as nn
from torchvision import models, transforms
from torch.utils.data import DataLoader, Subset, Dataset, WeightedRandomSampler
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from util import filter_data, seed_everything
from util import mask_crop as mask_crop_fn
from validate import val_model, ValLoaderWrapper
from loader import QSM_c1_Dataset as QSM_RAM_Dataset
from networks import QSMDecoder, ResNetWrapper
from datetime import datetime
import math

# ============================================================
# LORA IMPLEMENTATION
# ============================================================
class LoRALayer(nn.Module):
    def __init__(self, original_layer, rank=4, alpha=8):
        super().__init__()
        self.original_layer = original_layer
        self.rank = rank
        self.alpha = alpha
        self.scaling = alpha / rank
        
        # Determine device from the original layer
        dev = original_layer.weight.device
        
        # Freeze the original weights
        for param in self.original_layer.parameters():
            param.requires_grad = False
            
        if isinstance(original_layer, nn.Conv2d):
            out_channels, in_channels, k_h, k_w = original_layer.weight.shape
            # Initialize on the same device as original weights
            self.lora_A = nn.Parameter(torch.zeros(rank, in_channels, k_h, k_w, device=dev))
            self.lora_B = nn.Parameter(torch.zeros(out_channels, rank, 1, 1, device=dev))
            nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
            nn.init.zeros_(self.lora_B) 
        elif isinstance(original_layer, nn.Linear):
            out_f, in_f = original_layer.weight.shape
            self.lora_A = nn.Parameter(torch.zeros(rank, in_f, device=dev))
            self.lora_B = nn.Parameter(torch.zeros(out_f, rank, device=dev))
            nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
            nn.init.zeros_(self.lora_B)

    def forward(self, x):
        # x is on GPU, original_layer is on GPU, 
        # now lora_A/B are guaranteed to be on GPU
        orig_out = self.original_layer(x)
        if isinstance(self.original_layer, nn.Conv2d):
            lora_out = torch.nn.functional.conv2d(
                x, self.lora_A, 
                padding=self.original_layer.padding, 
                stride=self.original_layer.stride
            )
            lora_out = torch.nn.functional.conv2d(lora_out, self.lora_B)
        else:
            lora_out = (x @ self.lora_A.t()) @ self.lora_B.t()
        return orig_out + lora_out * self.scaling

# ============================================================
# EXPERIMENT CONFIG & NOISE CEILING SETUP
# ============================================================
EXP_NAME = "lora_noise_ceiling_v1" 
device = 'cuda:0'
LIMIT_SUBS = None 
CACHE_PATH = 'qsm_preprocessed_cache.pt'
LOAD_FROM_CACHE = True 
seed = 0
seed_everything(0)

TIMESTEPS = 100

def get_cosine_betas(timesteps, s=0.008):
    steps = timesteps + 1
    x = torch.linspace(0, timesteps, steps)
    alphas_cumprod = torch.cos(((x / timesteps) + s) / (1 + s) * torch.pi * 0.5) ** 2
    alphas_cumprod = alphas_cumprod / alphas_cumprod[0]
    betas = 1 - (alphas_cumprod[1:] / alphas_cumprod[:-1])
    return torch.clip(betas, 0.0001, 0.999)

betas = get_cosine_betas(TIMESTEPS).to(device)
alphas = 1. - betas
alphas_cumprod = torch.cumprod(alphas, dim=0)

def apply_diffusion_noise(x_0, t):
    noise = torch.randn_like(x_0)
    sqrt_alphas_cumprod_t = torch.sqrt(alphas_cumprod[t]).view(-1, 1, 1, 1)
    sqrt_one_minus_alphas_cumprod_t = torch.sqrt(1. - alphas_cumprod[t]).view(-1, 1, 1, 1)
    return sqrt_alphas_cumprod_t * x_0 + sqrt_one_minus_alphas_cumprod_t * noise

# ============================================================
# DATA PREPARATION
# ============================================================
nii_path = '/data2/ali/dbs/qsm/'
seg_path = '/data2/ali/dbs/seg_ps/'
file_dir = '/data2/ali/dbs/dbs_03292024.csv'
cv_features = {'Age', 'Sex', 'Ethnicity', 'Race', 'Disease Duration (year)', ' pre op levadopa equivalent dose (mg)', ' Test medication status', ' OFF (pre-dbs updrs)', ' ON (pre-dbs updrs)'}
all_needed_cols = cv_features | {'CORNELL ID', ' OFF meds ON stim 6mo'}
motor_df = filter_data(file_dir, all_needed_cols, True)
for col in [' OFF (pre-dbs updrs)', ' ON (pre-dbs updrs)', ' OFF meds ON stim 6mo']:
    motor_df[col] = pd.to_numeric(motor_df[col], errors='coerce')
motor_df = motor_df.dropna(subset=[' OFF (pre-dbs updrs)', ' OFF meds ON stim 6mo'])
improvement_ratios = (motor_df[' OFF (pre-dbs updrs)'] - motor_df[' OFF meds ON stim 6mo']) / motor_df[' OFF (pre-dbs updrs)']
label_map = {str(int(row['CORNELL ID'])): (1 if ratio >= 0.30 else 0) for (_, row), ratio in zip(motor_df.iterrows(), improvement_ratios)}
cols_to_norm = ['Age', 'Disease Duration (year)', ' OFF (pre-dbs updrs)', ' pre op levadopa equivalent dose (mg)']
for col in cols_to_norm:
    motor_df[col] = pd.to_numeric(motor_df[col], errors='coerce')
    col_mean, col_std = motor_df[col].mean(), motor_df[col].std()
    motor_df[col] = (motor_df[col] - col_mean) / (col_std + 1e-8)
clinical_dict = {str(int(row['CORNELL ID'])): row[list(cv_features)].values.astype(np.float32) for _, row in motor_df.iterrows()}
full_dataset = QSM_RAM_Dataset(nii_path, seg_path, mask_crop_fn, clinical_dict, label_map, limit=LIMIT_SUBS, cache_path=CACHE_PATH, load_cache=LOAD_FROM_CACHE, return_index=True)
actual_clin_dim = next(iter(clinical_dict.values())).shape[0]
full_dataset.clin_dim = actual_clin_dim
all_cached_ids = {str(k) for k in full_dataset.volumes.keys()}
label_keys = {str(k) for k in label_map.keys()}
unique_labeled_subs = np.array(sorted(list(all_cached_ids & label_keys)))
unique_sub_labels = np.array([label_map[sid] for sid in unique_labeled_subs])
unlabeled_ids = np.array(list(all_cached_ids - label_keys))

class RandomMasking(object):
    def __init__(self, mask_size=8, num_masks=2, p=0.5):
        self.mask_size = mask_size
        self.num_masks = num_masks
        self.p = p

    def __call__(self, tensor):
        if torch.rand(1).item() > self.p: return tensor
        _, h, w = tensor.shape
        for _ in range(self.num_masks):
            y = torch.randint(0, h - self.mask_size, (1,))
            x = torch.randint(0, w - self.mask_size, (1,))
            tensor[:, y:y+self.mask_size, x:x+self.mask_size] = 0
        return tensor

qsm_aug = transforms.Compose([
    transforms.RandomRotation(degrees=15),
    transforms.RandomAffine(degrees=0, translate=(0.05, 0.05), scale=(0.95, 1.05)),
    RandomMasking(p=0.3)
])

# ============================================================
# CROSS-VALIDATION LOOP
# ============================================================
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
log_filename = f"csdiff_lora_log_{timestamp}.txt"

def log_print(message):
    print(message)
    with open(log_filename, "a") as f:
        f.write(message + "\n")

log_print(f"Logging to: {log_filename}")

all_split_best_metrics = []
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)

for split, (t_p_idx, v_p_idx) in enumerate(skf.split(unique_labeled_subs, unique_sub_labels)):
    train_subs, val_subs = unique_labeled_subs[t_p_idx], unique_labeled_subs[v_p_idx]
    t_idx = [i for i, s in enumerate(full_dataset.samples) if str(s['sub_id']) in set(train_subs)]
    v_idx = [i for i, s in enumerate(full_dataset.samples) if str(s['sub_id']) in set(val_subs)]
    pt_subs = np.concatenate([unlabeled_ids, train_subs])
    pt_idx = [i for i, s in enumerate(full_dataset.samples) if str(s['sub_id']) in set(pt_subs)]

    train_labels = [label_map[str(full_dataset.samples[i]['sub_id'])] for i in t_idx]
    class_counts = np.bincount(train_labels)
    class_weights = 1. / torch.tensor(class_counts, dtype=torch.float)
    sampler = WeightedRandomSampler([class_weights[l] for l in train_labels], 2*len(t_idx))

    pt_loader = DataLoader(Subset(full_dataset, pt_idx), batch_size=48, shuffle=True)
    t_loader = DataLoader(Subset(full_dataset, t_idx), batch_size=48, sampler=sampler)
    v_loader = DataLoader(Subset(full_dataset, v_idx), batch_size=48, shuffle=False)

    base_resnet = models.resnet18(weights='IMAGENET1K_V1')
    base_resnet.conv1 = nn.Conv2d(1, 64, 7, 2, 3, bias=False)
    model = ResNetWrapper(base_resnet, clinical_dim=actual_clin_dim).to(device)
    decoder = QSMDecoder(feat_dim=512).to(device)

    # --- A. DIFFUSION PRETRAINING ---
    optimizer_pt = torch.optim.Adam(list(model.base_model.parameters()) + list(decoder.parameters()), lr=1e-4)
    criterion_pt = nn.MSELoss()
    full_dataset.train_mode, full_dataset.transform = True, qsm_aug
    for pt_epoch in range(10):
        model.base_model.train(); decoder.train()
        for imgs, clin, _, _ in pt_loader:
            imgs, clin = imgs.to(device), clin.to(device)
            t = torch.randint(0, TIMESTEPS, (imgs.shape[0],), device=device).long()
            noisy_imgs = apply_diffusion_noise(imgs, t)
            optimizer_pt.zero_grad()
            _, feats = model(noisy_imgs, clin)
            recon = decoder(feats)
            criterion_pt(recon, imgs).backward(); optimizer_pt.step()

    # --- B. FINE-TUNING (LoRA Injection) ---
    # Freeze all ResNet params first
    for param in model.base_model.parameters(): param.requires_grad = False
    
    # Replace Conv2d in Layer 3 and 4 with LoRA variants
    for layer_name in ["layer3", "layer4"]:
        target_layer = getattr(model.base_model, layer_name)
        for block_idx, block in enumerate(target_layer):
            if hasattr(block, "conv1"):
                block.conv1 = LoRALayer(block.conv1, rank=4)
            if hasattr(block, "conv2"):
                block.conv2 = LoRALayer(block.conv2, rank=4)
            # Handle downsample layers in blocks
            if block.downsample is not None:
                if isinstance(block.downsample[0], nn.Conv2d):
                    block.downsample[0] = LoRALayer(block.downsample[0], rank=4)

    # Optimizer: Fusion head + LoRA params
    lora_params = [p for n, p in model.named_parameters() if "lora_" in n]
    optimizer = torch.optim.Adam([
        {'params': lora_params, 'lr': 1e-4},
        {'params': model.fusion.parameters(), 'lr': 5e-5}
    ], weight_decay=1e-3)
    
    START_T, END_T, MAX_EPOCHS = 50, 10, 100 
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=MAX_EPOCHS, eta_min=1e-7)
    loss_fn = nn.CrossEntropyLoss().to(device)
    
    best_score, best_metrics_this_split, patience = 0, None, 0
    os.makedirs(f"weights/{EXP_NAME}", exist_ok=True)

    for epoch in range(MAX_EPOCHS):
        model.train(); full_dataset.train_mode = True
        
        if epoch < 10:
            current_max_t = START_T
        else:
            progress = (epoch - 10) / (MAX_EPOCHS - 10)
            current_max_t = int(START_T - (START_T - END_T) * progress)
        
        for imgs, clin, lbls, _ in t_loader:
            imgs, clin, lbls = imgs.to(device), clin.to(device), lbls.to(device)
            t_val = START_T if torch.rand(1).item() < 0.20 else current_max_t
            if t_val > 1:
                jitter = torch.randint(-3, 4, (1,)).item()
                t_limit = max(2, min(TIMESTEPS-1, t_val + jitter))
                t = torch.randint(1, t_limit, (imgs.shape[0],), device=device).long()
                noisy_imgs = apply_diffusion_noise(imgs, t)
            else:
                noisy_imgs = imgs
            
            optimizer.zero_grad()
            if torch.rand(1).item() < 0.5:
                lam = np.random.beta(0.2, 0.2)
                index = torch.randperm(imgs.size(0)).to(device)
                mixed_imgs = lam * noisy_imgs + (1 - lam) * noisy_imgs[index]
                mixed_clin = lam * clin + (1 - lam) * clin[index]
                logits, _ = model(mixed_imgs, mixed_clin)
                loss = lam * loss_fn(logits, lbls) + (1 - lam) * loss_fn(logits, lbls[index])
            else:
                logits, _ = model(noisy_imgs, clin)
                loss = loss_fn(logits, lbls)
            loss.backward(); optimizer.step()
        
        scheduler.step()
        model.eval(); full_dataset.train_mode, full_dataset.transform = False, None
        wrapped_v_loader = ValLoaderWrapper(v_loader)
        m = val_model(wrapped_v_loader, device, model, loss_fn, v_loader.dataset, threshold=0.5)
        
        current_acc, current_prec, current_sens, current_spec, current_auc = m[1], m[2], m[3], m[4], m[5]
        current_f1 = 2*(current_prec * current_sens)/(current_prec + current_sens) if (current_prec + current_sens) > 0 else 0
        current_score = (current_sens * current_spec * current_auc) ** (1/3)
        
        log_print(f"S{split} E{epoch} | MaxT: {current_max_t} | Acc: {current_acc:.4f} | Prec: {current_prec:.4f} | Sens: {current_sens:.4f} | Spec: {current_spec:.4f} | AUC: {current_auc:.4f} | F1: {current_f1:.4f}")

        if current_score > best_score:
            best_score, best_metrics_this_split, patience = current_score, m, 0
            torch.save(model.state_dict(), f"weights/{EXP_NAME}/split_{split}.pth")
        else:
            patience += 1

        if patience >= 30: 
            log_print(f"Early stopping triggered for Split {split}")
            break
    
    if best_metrics_this_split is not None:
        all_split_best_metrics.append(best_metrics_this_split)

# ============================================================
# FINAL SUMMARY
# ============================================================
final_metrics = np.array(all_split_best_metrics)
avg_metrics, std_metrics = np.mean(final_metrics, axis=0), np.std(final_metrics, axis=0)
print("\n" + "="*45 + "\nLORA + NOISE CEILING RESULTS\n" + "="*45)
names = ["Loss", "Accuracy", "Precision", "Sensitivity", "Specificity", "AUC"]
for i, name in enumerate(names):
    print(f"{name:<15} : {avg_metrics[i]:.4f} ± {std_metrics[i]:.4f}")
print("="*45)

Keeping CORNELL ID
Keeping Age
Keeping Sex
Keeping Ethnicity
Keeping Race
Keeping Disease Duration (year)
Keeping  OFF (pre-dbs updrs)
Keeping  ON (pre-dbs updrs)
Keeping  pre op levadopa equivalent dose (mg)
Keeping  Test medication status
Keeping  OFF meds ON stim 6mo
Loaded cache with 7776 slices from 108 subjects
Logging to: csdiff_lora_log_20260216_100140.txt
S0 E0 | MaxT: 50 | Acc: 0.7222 | Prec: 0.5244 | Sens: 0.2986 | Spec: 0.8917 | AUC: 0.6854 | F1: 0.3805
S0 E1 | MaxT: 50 | Acc: 0.7093 | Prec: 0.4926 | Sens: 0.5799 | Spec: 0.7611 | AUC: 0.6952 | F1: 0.5327
S0 E2 | MaxT: 50 | Acc: 0.7500 | Prec: 0.5629 | Sens: 0.5590 | Spec: 0.8264 | AUC: 0.7033 | F1: 0.5610
S0 E3 | MaxT: 50 | Acc: 0.6994 | Prec: 0.4828 | Sens: 0.7326 | Spec: 0.6861 | AUC: 0.7073 | F1: 0.5821
S0 E4 | MaxT: 50 | Acc: 0.7262 | Prec: 0.5171 | Sens: 0.6285 | Spec: 0.7653 | AUC: 0.7156 | F1: 0.5674
S0 E5 | MaxT: 50 | Acc: 0.7440 | Prec: 0.5469 | Sens: 0.6076 | Spec: 0.7986 | AUC: 0.7191 | F1: 0.5757
S0 E6 | MaxT: 5